# `ikbel_01` — Matching experiments (Ikbel)

Experiments for `oppradar_ai.matching` — weights, similarity backends, Precision@5 / nDCG@5 on `data/eval/match_test_set.jsonl`.

**Convention:** notebooks are prefixed `ikbel_<nn>_` so Ahlem and Ikbel never collide.

In [ ]:
import json
from pathlib import Path

from oppradar_ai.schemas import Opportunity, Profile
from oppradar_ai.matching import score_matches

fixtures = [Opportunity.model_validate(o) for o in json.loads(Path("../data/fixtures/opportunities.json").read_text(encoding="utf-8"))]
profiles = {
    p["id"]: Profile.model_validate(p)
    for p in map(json.loads, Path("../data/eval/profiles").glob("*.json"))
    for p in [p]
}
labels = [json.loads(l) for l in Path("../data/eval/match_test_set.jsonl").read_text(encoding="utf-8").splitlines()]
profiles.keys()

In [ ]:
# Top-5 per profile with real embeddings (needs LLM_API_KEY), or EMBEDDING_BACKEND=local.
for pid, profile in profiles.items():
    print(f"\n=== {pid} — {profile.full_name} ===")
    for m in score_matches(profile, fixtures, top_k=5):
        print(f"{m.score:.2f}  {m.opportunity.title[:45]:45s}  {'; '.join(m.reasons[:2])}")

In [ ]:
# Precision@5 per profile against the labelled triples (skips unlabelled items).
def precision_at_k(pid, k=5):
    gold = {l["opportunity_id"]: l["relevance"] for l in labels if l["profile_id"] == pid}
    ranked = score_matches(profiles[pid], fixtures, top_k=k)
    hits = sum(1 for m in ranked if gold.get(m.opportunity.id, 0) >= 2)
    return hits / k

{pid: round(precision_at_k(pid), 2) for pid in profiles}